## Importation des bibliothéques

In [1]:
# Pour la gestion des données et des opérations mathématiques
import numpy as np
import pandas as pd

# Pour l'affichage des images et la visualisation
import matplotlib.pyplot as plt

# Pour le traitement d'images (redimensionner, prétraitement, etc.)
import cv2
import os

# Pour le deep learning
import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Conv2D, MaxPooling2D, Flatten, Dense, Dropout
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from tensorflow.keras.callbacks import EarlyStopping
from tensorflow.keras.preprocessing.image import ImageDataGenerator



# Pour diviser le dataset en ensembles d'entraînement, de validation, et de test
from sklearn.model_selection import train_test_split

# Pour l'évaluation du modèle (confusion matrix, classification report)
from sklearn.metrics import classification_report, confusion_matrix






## Importation des images

In [2]:

# Chemin vers le dossier contenant les sous-dossiers d'émotions
dataset_dir = 'C:\\Users\\hp\\Desktop\\Innovation et plannification\\Data\\Final_DataSet'

# Créer un générateur d'images
datagen = ImageDataGenerator(rescale=1./255)  # Redimensionner les pixels entre 0 et 1

# Charger les données à partir des dossiers
train_data = datagen.flow_from_directory(
    dataset_dir, 
    target_size=(224, 224),  # Redimensionner les images à une taille standard
    batch_size=32,           # Nombre d'images dans un batch
    class_mode='categorical',  # Pour les étiquettes multiples (émotions)
    shuffle=True,            # Mélanger les images
    seed=42                  # Pour la reproductibilité
)


Found 29989 images belonging to 6 classes.


<table border="1">
  <tr>
    <th>Emotion</th>
    <th>Count</th>
  </tr>
  <tr>
    <td>happy</td>
    <td>5000</td>
  </tr>
  <tr>
    <td>nodesire</td>
    <td>5000</td>
  </tr>
  <tr>
    <td>surprise</td>
    <td>5000</td>
  </tr>
  <tr>
    <td>fatigue</td>
    <td>1243</td>
  </tr>
  <tr>
    <td>interest</td>
    <td>1233</td>
  </tr>
  <tr>
    <td>neutral</td>
    <td>5000</td>
  </tr>
</table>


Comment mentionner dans le tableau on a un désiquilibre dans les classes.
C'est pourquoi on va utiliser Data Augmentation, pour augmenter le nombre des images dans les classes minoritaires

In [3]:
import os
import numpy as np
from tensorflow.keras.preprocessing.image import ImageDataGenerator, img_to_array, array_to_img, load_img


In [ ]:
# Créer un générateur d'images pour l'augmentation des données
datagen_aug = ImageDataGenerator(
    rescale=1./255,  # Redimensionner les pixels entre 0 et 1
    rotation_range=10,  # Rotation aléatoire entre -30 et 30 degrés
    width_shift_range=0.2,  # Translation horizontale aléatoire
    height_shift_range=0.2,  # Translation verticale aléatoire
    shear_range=0.2,  # Transformation de cisaillement
    zoom_range=0.2,  # Zoom aléatoire
    horizontal_flip=True,  # Retourner les images horizontalement
    fill_mode='nearest'  # Mode de remplissage pour les pixels manquants
)

In [ ]:
# Définir le nombre d'images que l'on veut atteindre pour chaque classe minoritaire
#target_count = 5000

In [ ]:
# Fonction pour générer et sauvegarder les images augmentées
"""def generate_augmented_images(class_name, current_count, class_dir, target_count=5000):
    class_path = os.path.join(class_dir, class_name)
    output_path = os.path.join(class_path, 'augmented')
    
    # Créer le dossier pour les images augmentées si ce n'est pas déjà fait
    if not os.path.exists(output_path):
        os.makedirs(output_path)
    
    # Charger les images de la classe
    images = []
    for img_name in os.listdir(class_path):
        if img_name.endswith('.jpg') or img_name.endswith('.png'):
            try:
                img = load_img(os.path.join(class_path, img_name), target_size=(224, 224))
                img = img_to_array(img)
                images.append(img)
            except Exception as e:
                print(f"Erreur lors du chargement de l'image {img_name}: {e}")
    
    if len(images) == 0:
        print(f"Aucune image valide dans le dossier {class_name}")
        return

    # Convertir la liste en tableau numpy
    images = np.array(images)
    
    # Créer un générateur d'augmentation des images
    datagen_aug.fit(images)
    
    # Générer et sauvegarder les nouvelles images
    for i, img in enumerate(datagen_aug.flow(images, batch_size=1, save_to_dir=output_path, save_prefix='aug', save_format='jpeg')):
        current_count += 1
        if current_count >= target_count:
            break
    print(f"Augmentation terminée pour la classe {class_name}, {current_count} images créées.")"""

In [ ]:
# Appliquer l'augmentation pour chaque classe minoritaire (Fatigue, Interest)
#generate_augmented_images('fatigue', 1243, dataset_dir)


Augmentation terminée pour la classe fatigue, 5000 images créées.


In [ ]:
#generate_augmented_images('Interest', 1233, dataset_dir)


Augmentation terminée pour la classe Interest, 5000 images créées.


Les images augmentées sont en format jpeg, or les images initiales sont en format jpg, alors on va les convertir de jpeg vers jpg :

In [ ]:
"""import os

# Fonction pour renommer les fichiers d'images de .jpeg à .jpg dans le dossier spécifié
def rename_images_in_directory(directory):
    for filename in os.listdir(directory):
        if filename.endswith('.jpeg'):
            # Nouveau nom de fichier avec l'extension .jpg
            new_filename = filename.replace('.jpeg', '.jpg')
            old_filepath = os.path.join(directory, filename)
            new_filepath = os.path.join(directory, new_filename)
            # Renommer le fichier
            os.rename(old_filepath, new_filepath)
            print(f"Renommé {filename} en {new_filename}")

# Dossiers contenant les images augmentées pour chaque classe
classes = ['fatigue', 'Interest']

# Parcourir chaque classe et renommer les fichiers dans le dossier 'augmented'
for class_name in classes:
    augmented_dir = os.path.join(dataset_dir, class_name, 'augmented')
    if os.path.exists(augmented_dir):
        rename_images_in_directory(augmented_dir)
    else:
        print(f"Dossier augmenté non trouvé pour la classe {class_name}: {augmented_dir}")"""


Renommé aug_0_2485.jpeg en aug_0_2485.jpg
Renommé aug_0_581.jpeg en aug_0_581.jpg
Renommé aug_0_6659.jpeg en aug_0_6659.jpg
Renommé aug_1000_6601.jpeg en aug_1000_6601.jpg
Renommé aug_1000_7111.jpeg en aug_1000_7111.jpg
Renommé aug_1000_83.jpeg en aug_1000_83.jpg
Renommé aug_1001_8901.jpeg en aug_1001_8901.jpg
Renommé aug_1001_904.jpeg en aug_1001_904.jpg
Renommé aug_1001_9359.jpeg en aug_1001_9359.jpg
Renommé aug_1002_3002.jpeg en aug_1002_3002.jpg
Renommé aug_1002_3932.jpeg en aug_1002_3932.jpg
Renommé aug_1002_4203.jpeg en aug_1002_4203.jpg
Renommé aug_1003_3840.jpeg en aug_1003_3840.jpg
Renommé aug_1003_6900.jpeg en aug_1003_6900.jpg
Renommé aug_1003_7415.jpeg en aug_1003_7415.jpg
Renommé aug_1004_4653.jpeg en aug_1004_4653.jpg
Renommé aug_1004_6382.jpeg en aug_1004_6382.jpg
Renommé aug_1004_9725.jpeg en aug_1004_9725.jpg
Renommé aug_1005_1542.jpeg en aug_1005_1542.jpg
Renommé aug_1005_370.jpeg en aug_1005_370.jpg
Renommé aug_1005_4673.jpeg en aug_1005_4673.jpg
Renommé aug_1006_450

Remarque:

Copier les images augmentées vers le dossier de la classe

<table border="1">
  <tr>
    <th>Emotion</th>
    <th>Count</th>
  </tr>
  <tr>
    <td>happy</td>
    <td>5000</td>
  </tr>
  <tr>
    <td>nodesire</td>
    <td>5000</td>
  </tr>
  <tr>
    <td>surprise</td>
    <td>5000</td>
  </tr>
  <tr>
    <td>fatigue</td>
    <td>5000</td>
  </tr>
  <tr>
    <td>interest</td>
    <td>5000</td>
  </tr>
  <tr>
    <td>neutral</td>
    <td>5000</td>
  </tr>
</table>


# Dévision des données en train, validation and test

Pour ce faire, On diviser les données en 3 dossiers :

train 'Final_DataSet: contenant 5000 images pour chaque classe

test 'Final_DataSet_Test': contenant 107 images pour chaque classe

val 'Final_DataSet_Val': contenant 107 images pour chaque classe


## La base de données a été construite avec succés!!!

<pre>Final_DataSet/
├── nodesire/
│   ├── image1.jpg
│   ├── image2.jpg
│   └── ...
├── happy/
│   ├── image1.jpg
│   ├── image2.jpg
│   └── ...
├── neutral/
│   ├── image1.jpg
│   ├── image2.jpg
│   └── ...
├── interest/
│   ├── image1.jpg
│   ├── image2.jpg
│   └── ...
├── fatigue/
│   ├── image1.jpg
│   ├── image2.jpg
│   └── ...
└── surprise/
    ├── image1.jpg
    ├── image2.jpg
    └── ...

Final_DataSet_Test/
├── nodesire/
│   ├── image1.jpg
│   ├── image2.jpg
│   └── ...
├── happy/
│   ├── image1.jpg
│   ├── image2.jpg
│   └── ...
├── neutral/
│   ├── image1.jpg
│   ├── image2.jpg
│   └── ...
├── interest/
│   ├── image1.jpg
│   ├── image2.jpg
│   └── ...
├── fatigue/
│   ├── image1.jpg
│   ├── image2.jpg
│   └── ...
└── surprise/
    ├── image1.jpg
    ├── image2.jpg
    └── ...

Final_DataSet_Val/
├── nodesire/
│   ├── image1.jpg
│   ├── image2.jpg
│   └── ...
├── happy/
│   ├── image1.jpg
│   ├── image2.jpg
│   └── ...
├── neutral/
│   ├── image1.jpg
│   ├── image2.jpg
│   └── ...
├── interest/
│   ├── image1.jpg
│   ├── image2.jpg
│   └── ...
├── fatigue/
│   ├── image1.jpg
│   ├── image2.jpg
│   └── ...
└── surprise/
    ├── image1.jpg
    ├── image2.jpg
    └── ...
</pre>